# 🛡️ DocuGuard+ — Google Colab GPU Backend

> **Run your LLM humanization on Colab's free T4 GPU (16 GB VRAM) and connect it to your local DocuGuard+ app via Cloudflare tunnel.**

## What This Notebook Does

| Step | Action |
|------|--------|
| 1 | Installs **Ollama** (LLM runtime) on the Colab VM |
| 2 | Installs **Cloudflare Tunnel** to expose the API publicly |
| 3 | Pulls your chosen LLM model (Mistral / LLama3) |
| 4 | Starts a **Cloudflare tunnel** → gives you a public URL |
| 5 | You paste that URL into DocuGuard+'s sidebar → done! |

## Prerequisites
- Google Colab account (free tier works)
- **Runtime → Change runtime type → T4 GPU**
- Your local machine running DocuGuard+ Streamlit app

## Performance
| Setup | Speed |
|-------|-------|
| Local CPU | ~5-15 tokens/sec |
| **Colab T4 GPU** | **~40-80 tokens/sec (5-8x faster)** |

## Step 1: Install Ollama & Cloudflare Tunnel
This installs the Ollama LLM runtime and the Cloudflare tunnel binary on the Colab VM.

In [ ]:
%%bash
# Install Ollama
curl -fsSL https://ollama.com/install.sh | sh

# Install Cloudflare tunnel
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
  -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared

echo "✅ Ollama and Cloudflare installed successfully"

## Step 2: Start Ollama Server (GPU-Accelerated)
Launches the Ollama server bound to `0.0.0.0:11434` so the tunnel can reach it.

In [ ]:
import subprocess, os, time

# Start Ollama server with GPU support
env = {**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"}
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

# Verify it's running
import requests
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print(f"✅ Ollama server is running (status {r.status_code})")
except Exception as e:
    print(f"❌ Server not ready yet: {e}")
    print("   Waiting 10 more seconds...")
    time.sleep(10)
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print(f"✅ Ollama server is running (status {r.status_code})")

## Step 3: Pull LLM Model

Choose your model:
- **`mistral`** — 4.1 GB, fast, good quality (recommended)
- **`llama3`** — 4.7 GB, slightly better quality, slower

Run **one** of the cells below.

In [ ]:
# ===== Option A: Mistral (Recommended — fast & effective) =====
!ollama pull mistral
print("✅ Mistral model ready")

In [ ]:
# ===== Option B: LLama 3 (Better quality, slightly slower) =====
# Uncomment and run this cell instead if you prefer LLama 3
# !ollama pull llama3
# print("✅ LLama 3 model ready")

## Step 4: Verify GPU Usage
Confirm the T4 GPU is detected and Ollama is using it.

In [ ]:
!echo "=== GPU Info ==="
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

!echo ""
!echo "=== Loaded Models ==="
!ollama list

!echo ""
!echo "=== Quick Inference Test ==="
import requests, time
start = time.time()
r = requests.post("http://localhost:11434/api/generate", json={
    "model": "mistral",
    "prompt": "Say hello in one sentence.",
    "stream": False,
})
elapsed = time.time() - start
resp = r.json().get("response", "")
print(f"Response: {resp[:200]}")
print(f"⏱️ Inference time: {elapsed:.2f}s")

## Step 5: Start Cloudflare Tunnel 🚀

This creates a **free public URL** that forwards to your Colab's Ollama server.

**Copy the URL** that appears below and paste it into DocuGuard+'s sidebar under **🔗 LLM Backend**.

In [ ]:
import subprocess, re, time, threading
from IPython.display import display, Markdown, HTML

# Start Cloudflare tunnel in background
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait for the tunnel URL to appear
tunnel_url = None
deadline = time.time() + 30

while time.time() < deadline:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r"(https://[a-z0-9-]+\.trycloudflare\.com)", line)
    if match:
        tunnel_url = match.group(1)
        break

if tunnel_url:
    display(HTML(f"""
    <div style="background: #1a1a2e; border: 3px solid #00d4ff; border-radius: 12px;
                padding: 20px; margin: 10px 0; text-align: center;">
        <h2 style="color: #00d4ff; margin: 0 0 10px 0;">🚀 Tunnel Active!</h2>
        <p style="color: #e0e0e0; font-size: 14px; margin: 5px 0;">
            Copy this URL and paste it into DocuGuard+ sidebar → <b>🔗 LLM Backend</b>
        </p>
        <code style="background: #0d1117; color: #58a6ff; padding: 12px 24px;
                    border-radius: 8px; font-size: 18px; display: inline-block;
                    margin: 10px 0; user-select: all; cursor: pointer;">
            {tunnel_url}
        </code>
        <p style="color: #8b949e; font-size: 12px; margin: 10px 0 0 0;">
            ⚠️ This URL is temporary and changes each time you restart.
            Keep this notebook running!
        </p>
    </div>
    """))
else:
    print("❌ Tunnel failed to start within 30 seconds.")
    print("Check logs: !cat /tmp/ollama.log")

## Step 6: Keep Alive ♾️

**Run this cell and leave it running.** It keeps the Colab session and tunnel alive.

The session will auto-disconnect after ~90 minutes of inactivity on free tier.
To extend, interact with the notebook periodically.

In [ ]:
import time, requests
from IPython.display import clear_output

print("♾️ Keep-alive running. Do NOT stop this cell.")
print(f"🔗 Tunnel URL: {tunnel_url}")
print("-" * 50)

elapsed_min = 0
while True:
    time.sleep(60)
    elapsed_min += 1

    # Heartbeat check
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        status = f"✅ Ollama OK ({len(r.json().get('models', []))} models)"
    except Exception:
        status = "⚠️ Ollama not responding"

    # Print periodic status (every 5 minutes)
    if elapsed_min % 5 == 0:
        print(f"[{elapsed_min}min] {status} | Tunnel: {tunnel_url}")